In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
import os
import glob
import re
import gc

In [32]:
# 0) OPTIONAL ▸ set to 1031 or [1031, 1142] to limit participants
participants_to_load = None
# 1) WHAT TO LOAD – just edit this list
sessions_to_load = [1]          # e.g. [1, 2, 3...] to load several sessions

# 2) WHERE YOUR RAW CSVs LIVE
raw_folder = r"C:/Users/Marcel/Desktop/Study Project/VR Data/Raw Data All Participants"

# 3) WHERE TO SAVE THE PROCESSED FILES (created if absent)
base_folder = os.path.dirname(raw_folder)         
out_folder  = os.path.join(base_folder, "Angular Velocities Processed")
os.makedirs(out_folder, exist_ok=True)

# LOAD ONLY THE DESIRED SESSION FILES
pattern = re.compile(r'(\d{4})_(\d)\.csv$')     # matches the 1234_1.csv format
dfs = []  # will collect DataFrames here

for filepath in glob.glob(os.path.join(raw_folder, '*.csv')):
    name = os.path.basename(filepath)
    m = pattern.match(name)
    if not m:
        continue  # skip anything that doesn’t fit ####_#.csv

    pid, sess = map(int, m.groups())

    # --- participant filter -------------------------------------------
    if participants_to_load is not None:
        if isinstance(participants_to_load, (list, tuple, set)):
            if pid not in participants_to_load:
                continue
        else:  # single int
            if pid != participants_to_load:
                continue

    # --- session filter ------------------------------------------------
    if sess not in sessions_to_load:
        continue

    # --- load ----------------------------------------------------------
    df = pd.read_csv(filepath, sep=",")
    dfs.append(df)

# SUMMARY OF WHAT GOT LOADED
print(
    f"Loaded {len(dfs)} file(s) for session(s) {sessions_to_load} "
    f"{'and participants ' + str(participants_to_load) if participants_to_load else '(all participants)'}."
)


Loaded 29 file(s) for session(s) [1] (all participants).


In [33]:
# merge 'hmdDirectionRight.x -.y and -.z into each raw dataframe already stored in `dfs`
HMD_FOLDER = r"C:/Users/Marcel/Desktop/Study Project/05_Debbies_gaze"
TIME_COL   = "timeStampDataPointEnd"
HMD_RIGHT_COLS = ["hmdDirectionRight.x", "hmdDirectionRight.y", "hmdDirectionRight.z"]

merged_dfs = []

for df in dfs:
    pid  = int(df["SubjectID"].iat[0])
    sess = int(df["Session"].iat[0])

    if sess not in sessions_to_load:
        merged_dfs.append(df)
        continue

    hmd_path = os.path.join(HMD_FOLDER, f"{pid:04d}_{sess}.csv")
    if not os.path.isfile(hmd_path):
        print(f"[missing] {hmd_path}")
        merged_dfs.append(df)
        continue

    # Read only the time column + HMD right columns
    try:
        df_hmd = pd.read_csv(hmd_path, usecols=[TIME_COL] + HMD_RIGHT_COLS)
    except ValueError as e:
        raise ValueError(f"Expected columns not found in {hmd_path}: {e}")

    # Merge by exact time, keeping all rows from df
    before_cols = df.shape[1]
    df_merged = df.merge(df_hmd, on=TIME_COL, how="left")  # matches only where time is identical
    after_cols = df_merged.shape[1]

    matched_rows = df_merged[HMD_RIGHT_COLS].notna().all(axis=1).sum()
    total_rows = len(df)

    print(f"{pid:04d}_{sess}: cols {before_cols} → {after_cols} "
          f"(rows unchanged {total_rows:,}) | matched {matched_rows:,}/{total_rows:,} rows by time")

    merged_dfs.append(df_merged)

# Replace dfs with enriched versions
dfs = merged_dfs

1031_1: cols 48 → 51 (rows unchanged 111,696) | matched 111,696/111,696 rows by time
1142_1: cols 48 → 51 (rows unchanged 116,095) | matched 116,095/116,095 rows by time
1234_1: cols 48 → 51 (rows unchanged 111,013) | matched 111,013/111,013 rows by time
1268_1: cols 48 → 51 (rows unchanged 113,819) | matched 113,819/113,819 rows by time
1574_1: cols 48 → 51 (rows unchanged 115,808) | matched 115,808/115,808 rows by time
1843_1: cols 48 → 51 (rows unchanged 114,955) | matched 114,955/114,955 rows by time
2069_1: cols 48 → 51 (rows unchanged 115,258) | matched 115,258/115,258 rows by time
3193_1: cols 48 → 51 (rows unchanged 113,013) | matched 113,013/113,013 rows by time
3540_1: cols 48 → 51 (rows unchanged 117,006) | matched 117,006/117,006 rows by time
4580_1: cols 48 → 51 (rows unchanged 77,347) | matched 77,347/77,347 rows by time
4598_1: cols 48 → 51 (rows unchanged 116,902) | matched 116,902/116,902 rows by time
4847_1: cols 48 → 51 (rows unchanged 112,899) | matched 112,899/112,

In [ ]:
# --- Config ---
RIGHT_COLS  = ["hmdDirectionRight.x","hmdDirectionRight.y","hmdDirectionRight.z"]
GAP_FACTOR  = 2.0    # interval is "big gap" if dt > GAP_FACTOR * median(dt)
SG_WIN      = 5      # odd, >=5
SG_POLY     = 2
VEL_CAP_MAX = 800.0  # deg/s

# --- Helpers ---
def _normalize_rows(v):
    v = np.asarray(v, float)
    n = np.linalg.norm(v, axis=1, keepdims=True)
    n[n == 0.0] = 1.0
    return v / n

def _smooth_finite_runs(x, win, poly):
    """Savitzky-Golay within contiguous finite runs; do NOT smooth across NaNs/gaps."""
    x = np.asarray(x, float)
    out = x.copy()
    finite = np.isfinite(x)
    if finite.sum() < 3:
        return out
    idx = np.flatnonzero(finite)
    cuts = np.where(np.diff(idx) > 1)[0] + 1
    for seg in np.split(idx, cuts):
        n = len(seg)
        if n < 3:
            continue
        max_odd = n if (n % 2 == 1) else (n - 1)
        w = min(win, max_odd)
        if w <= poly:
            need = poly + 1 if ((poly + 1) % 2 == 1) else (poly + 2)
            if need > n:
                continue
            w = min(need, max_odd)
        out[seg] = savgol_filter(out[seg], window_length=w, polyorder=poly, mode="interp")
    return out

# -------- Main loop: compute head velocity using forward Δt from total_time --------
for df in dfs:
    # build absolute time from per-row shifts (in seconds)
    df["total_time"] = df["Time_Shift"].cumsum()
    # Sort by time
    df.sort_values("total_time", inplace=True, kind="mergesort", ignore_index=True)

    t = df["total_time"].to_numpy(float)
    N = len(t)

    # Forward Δt
    dt_fwd = np.diff(t)  # length N-1
    dt_valid = dt_fwd[(dt_fwd > 0) & np.isfinite(dt_fwd)]
    dt_med = np.nanmedian(dt_valid) if dt_valid.size else np.nan

    # Big gaps: exclude Time_Shift gaps
    big_gap_intervals = (dt_fwd > (GAP_FACTOR * dt_med)) if np.isfinite(dt_med) else np.zeros_like(dt_fwd, dtype=bool)
    bad_interval = (~np.isfinite(dt_fwd)) | (dt_fwd <= 0) | big_gap_intervals

    # Head direction unit vectors, angles between consecutive samples
    r = _normalize_rows(df[RIGHT_COLS].to_numpy(float))     
    dots = np.einsum("ij,ij->i", r[:-1], r[1:])              
    dots = np.clip(dots, -1.0, 1.0)
    dtheta = np.arccos(dots)                                 # radians

    # Velocity assigned to row i for interval (i -> i+1)
    vel = np.full(N, np.nan, float)
    good = (~bad_interval) & np.isfinite(dtheta)
    vel[:-1][good] = np.degrees(dtheta[good]) / dt_fwd[good]  # deg/s

    # save the raw, unsmoothed, unclipped series
    df["head_velocity_unsmoothed_deg_per_s"] = vel
    
    # Smooth within finite runs, then clip to [0, 800]
    vel_s = _smooth_finite_runs(vel, SG_WIN, SG_POLY)
    vel_mag = np.clip(vel_s, 0.0, VEL_CAP_MAX)

    # Write ONLY the final series
    df["head_velocity_magnitude_deg_per_s"] = vel_mag

In [5]:
# merge eyeDirectionCombinedWorld.x into each raw dataframe already stored in `dfs`
GAZE_FOLDER      = r"C:/Users/Marcel/Desktop/Study Project/05_Debbies_gaze"
SESSIONS_TO_LOAD = [1]                    # edit as needed

merged_dfs = []

for df_raw in dfs:                        # iterate over the dataframes
    pid  = int(df_raw['SubjectID'].iat[0])
    sess = int(df_raw['Session']  .iat[0])

    if sess not in SESSIONS_TO_LOAD:
        merged_dfs.append(df_raw)
        continue

    gaze_path = os.path.join(GAZE_FOLDER, f"{pid:04d}_{sess}.csv")
    if not os.path.isfile(gaze_path):
        print(f"[missing] {gaze_path}")
        merged_dfs.append(df_raw)
        continue

    # read the three gaze columns (+ IDs)
    df_gaze = (
        pd.read_csv(
            gaze_path,
            usecols=['SubjectID', 'Session',
                     'eyeDirectionCombinedWorld.x',
                     'eyeDirectionCombinedWorld.y',
                     'eyeDirectionCombinedWorld.z'])
        .drop_duplicates(subset=['SubjectID', 'Session',
                                 'eyeDirectionCombinedWorld.y',
                                 'eyeDirectionCombinedWorld.z'])
    )

    df_merge = df_raw.merge(
        df_gaze,
        on=['SubjectID', 'Session',
            'eyeDirectionCombinedWorld.y', 'eyeDirectionCombinedWorld.z'],
        how='left'
    )

    # move 'eyeDirectionCombinedWorld.x' to the left of 'eyeDirectionCombinedWorld.y'
    if ('eyeDirectionCombinedWorld.x' in df_merge.columns and
        'eyeDirectionCombinedWorld.y' in df_merge.columns):
        df_merge.insert(
            df_merge.columns.get_loc('eyeDirectionCombinedWorld.y'),
            'eyeDirectionCombinedWorld.x',
            df_merge.pop('eyeDirectionCombinedWorld.x')
        )

    print(f"{pid:04d}_{sess}: cols {df_raw.shape[1]} → {df_merge.shape[1]} "
          f"(rows unchanged {df_raw.shape[0]:,})")

    merged_dfs.append(df_merge)

# replace dfs so downstream code uses the enriched frames
dfs = merged_dfs

1031_1: cols 53 → 54 (rows unchanged 111,696)
1142_1: cols 53 → 54 (rows unchanged 116,095)
1234_1: cols 53 → 54 (rows unchanged 111,013)
1268_1: cols 53 → 54 (rows unchanged 113,819)
1574_1: cols 53 → 54 (rows unchanged 115,808)
1843_1: cols 53 → 54 (rows unchanged 114,955)
2069_1: cols 53 → 54 (rows unchanged 115,258)
3193_1: cols 53 → 54 (rows unchanged 113,013)
3540_1: cols 53 → 54 (rows unchanged 117,006)
4580_1: cols 53 → 54 (rows unchanged 77,347)
4598_1: cols 53 → 54 (rows unchanged 116,902)
4847_1: cols 53 → 54 (rows unchanged 112,899)
4875_1: cols 53 → 54 (rows unchanged 111,305)
5161_1: cols 53 → 54 (rows unchanged 113,729)
5189_1: cols 53 → 54 (rows unchanged 113,542)
5191_1: cols 53 → 54 (rows unchanged 113,223)
5743_1: cols 53 → 54 (rows unchanged 114,256)
5766_1: cols 53 → 54 (rows unchanged 115,906)
5851_1: cols 53 → 54 (rows unchanged 113,845)
5972_1: cols 53 → 54 (rows unchanged 114,721)
6266_1: cols 53 → 54 (rows unchanged 112,785)
6406_1: cols 53 → 54 (rows unchange

In [ ]:
## PROCESS & SAVE EACH PARTICIPANT‑SESSION FILE SEPARATELY

## Movement Correction & Angular Velocity Calculation ##

'''
eye_pos => eyePositionCombinedWorld
hit => hitPointOnObject
time => timeStampDataPointEnd

'''

for df in dfs:
    pid  = int(df["SubjectID"].iat[0]) 
    sess = int(df["Session"].iat[0])
    
    # extract relevant columns (coordinates)
    eye_pos=list(zip(df["eyePositionCombinedWorld.x"],df["eyePositionCombinedWorld.y"],df["eyePositionCombinedWorld.z"]))
    hit_pos=list(zip(df["hitPointOnObject_x"],df["hitPointOnObject_y"],df["hitPointOnObject_z"]))

    # compute hit point displacement
    hit_delta = list(zip(
        df["hitPointOnObject_x"].shift(-1) - df["hitPointOnObject_x"],
        df["hitPointOnObject_y"].shift(-1) - df["hitPointOnObject_y"],
        df["hitPointOnObject_z"].shift(-1) - df["hitPointOnObject_z"],
    ))


    # Time differences
      # Measures how much time passed between samples. (forward diff)
    time_diff = df["timeStampDataPointEnd"].diff().shift(-1).tolist()

    # Gaze direction vectors (eye -> hit point)
      # gaze vector: direction from the eye to the hitpoint
      # unit gaze vector: Normalized (length = 1) to work in direction only.
    g_vec = [np.subtract(hit, eye) for hit, eye in zip(hit_pos, eye_pos)]
    unit_g_vec = [v / np.linalg.norm(v) if np.linalg.norm(v) != 0 else np.zeros(3)
                     for v in g_vec]

    # Project hit movement onto gaze direction
      # dot_products = scalar product
    dot_products = [np.dot(delta, gaze) for delta, gaze in zip(hit_delta, unit_g_vec)]
    projections = [dot_products[i] * unit_g_vec[i] for i in range(len(dot_products))]


    # Get orthogonal component (in-plane)
    inplane_vec = [np.subtract(hit_delta[i], projections[i]) for i in range(len(hit_delta))]
    inplane_length = [np.linalg.norm(vec) for vec in inplane_vec]

    # Distances between eye coordinates and hit on object   - eucledian distance
    eye_hit_dis = [np.linalg.norm(np.subtract(hit, eye)) for hit, eye in zip(hit_pos, eye_pos)]

    # Angular velocity (rad → deg/s)
    angular_velocity = []
    for i in range(len(inplane_length)):
        if time_diff[i] != 0 and eye_hit_dis[i] != 0:
            angle_rad = math.atan2(inplane_length[i], eye_hit_dis[i])
            angle_deg_per_s = angle_rad / time_diff[i] * (180 / math.pi)
        else:
            angle_deg_per_s = 0
        angular_velocity.append(angle_deg_per_s)

    df["angular_gaze_velocity_deg_per_s"] = angular_velocity

    #### Savitzky-Golay smoothing (keep your params) ####
    window_length = 3
    polyorder     = 2

    vel = df["angular_gaze_velocity_deg_per_s"].to_numpy(float)
    smoothed_unclipped = savgol_filter(vel, window_length, polyorder, mode="interp")
    # if too few points, just copy; else smooth
    if len(vel) >= window_length and window_length > polyorder:
        smoothed = savgol_filter(vel, window_length, polyorder, mode="interp")
    else:
        smoothed = vel.copy()

    df["angular_gaze_velocity_smoothed_unclipped"] = smoothed_unclipped

    # enforce biological limits 0-1000 deg/s (Nolte et al. 2024)
    df["angular_gaze_velocity_smoothed"] = np.clip(smoothed_unclipped, 0, 1000)  

    # drop last 2 rows since its NA by default
    df.drop(df.index[-2:], inplace=True)

    # save
    out_name = f"{pid:04d}_{sess}_wAngularVelocities.csv"
    df.to_csv(os.path.join(out_folder, out_name),
              index=False)
    print(f"Processed & saved {out_name}")

print("\nAll files done ✔️")



Processed & saved 1031_1_wAngularVelocities.csv
Processed & saved 1142_1_wAngularVelocities.csv
Processed & saved 1234_1_wAngularVelocities.csv
Processed & saved 1268_1_wAngularVelocities.csv
Processed & saved 1574_1_wAngularVelocities.csv
Processed & saved 1843_1_wAngularVelocities.csv
Processed & saved 2069_1_wAngularVelocities.csv
Processed & saved 3193_1_wAngularVelocities.csv
Processed & saved 3540_1_wAngularVelocities.csv
Processed & saved 4580_1_wAngularVelocities.csv
Processed & saved 4598_1_wAngularVelocities.csv
Processed & saved 4847_1_wAngularVelocities.csv
Processed & saved 4875_1_wAngularVelocities.csv
Processed & saved 5161_1_wAngularVelocities.csv
Processed & saved 5189_1_wAngularVelocities.csv
Processed & saved 5191_1_wAngularVelocities.csv
Processed & saved 5743_1_wAngularVelocities.csv
Processed & saved 5766_1_wAngularVelocities.csv
Processed & saved 5851_1_wAngularVelocities.csv
Processed & saved 5972_1_wAngularVelocities.csv
Processed & saved 6266_1_wAngularVelocit